# Assemble subset .h5ad files

For QC and inspection of our cell type labels, we'll assemble samples based on subject metadata to get some reusable units that will be manageable for downstream analysis. 

To each of these subsets, we'll add our cell type label predictions and scrublet scores and calls.

We'll carry these subsets forward into QC filtering and cell type-based doublet and mislabeling clean-up in later notebooks.

## Load Packages

`anndata`: Data structures for scRNA-seq  
`datetime`: date and time functions  
`h5py`: HDF5 file I/O  
`hisepy`: The HISE SDK for Python  
`os`: operating system calls  
`pandas`: DataFrame data structures  
`re`: Regular expressions  
`scanpy`: scRNA-seq analysis  
`scipy.sparse`: Spare matrix data structures  

In [21]:
import anndata
from datetime import date
import h5py
import hisepy
import os
import pandas as pd
from pandas.api.types import is_object_dtype
import re
import scanpy as sc
import scipy.sparse as scs
import numpy as np

## Helper functions

In [2]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [53]:
hisepy.__version__

'0.3.0'

In [3]:
def read_csv_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_csv(cache_file)
    return res

In [4]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

Functions to read pipeline .h5 files as anndata

In [5]:
# define a function to read count data
def read_mat(h5_con):
    mat = scs.csc_matrix(
        (h5_con['matrix']['data'][:], # Count values
         h5_con['matrix']['indices'][:], # Row indices
         h5_con['matrix']['indptr'][:]), # Pointers for column positions
        shape = tuple(h5_con['matrix']['shape'][:]) # Matrix dimensions
    )
    return mat

# define a function to read obeservation metadata (i.e. cell metadata)
def read_obs(h5con):
    bc = h5con['matrix']['barcodes'][:]
    bc = [x.decode('UTF-8') for x in bc]

    # Initialized the DataFrame with cell barcodes
    obs_df = pd.DataFrame({ 'barcodes' : bc })

    # Get the list of available metadata columns
    obs_columns = h5con['matrix']['observations'].keys()
    
    # For each column
    for col in obs_columns:
        # Read the values
        values = h5con['matrix']['observations'][col][:]
        # Check for byte storage
        if(isinstance(values[0], (bytes, bytearray))):
            # Decode byte strings
            values = [x.decode('UTF-8') for x in values]
        # Add column to the DataFrame
        obs_df[col] = values

    obs_df = obs_df.set_index('barcodes', drop = False)
    
    return obs_df

# define a function to construct anndata object from a h5 file
def read_h5_anndata(h5_file):
    h5_con = h5py.File(h5_file, mode = 'r')
    # extract the expression matrix
    mat = read_mat(h5_con)
    # extract gene names
    genes = h5_con['matrix']['features']['name'][:]
    genes = [x.decode('UTF-8') for x in genes]
    # extract metadata
    obs_df = read_obs(h5_con)
    # construct anndata
    adata = anndata.AnnData(mat.T,
                             obs = obs_df)
    # make sure the gene names aligned
    adata.var_names = genes

    adata.var_names_make_unique()
    return adata

In [6]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Read sample metadata from HISE

In [7]:
in_uuids = []

In [8]:
## sample_meta_uuid from previous step
sample_meta_uuid = '2b3673ee-827b-415e-837b-f27a0b88eae2'
sample_meta = read_csv_uuid(sample_meta_uuid)
in_uuids.append(sample_meta_uuid)

We only need to keep some of the metadata columns that pertain to cohort, subject, and sample. We'll also keep the originating File GUID to help us keep track of provenance. Let's select just these columns:

In [9]:
keep_meta = [
    'cohort.cohortGuid',
    'subject.subjectGuid', 'subject.biologicalSex', 
    'subject.race', 'subject.ethnicity', 'subject.birthYear',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawDate',
    'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit',
    'file.id'
]

In [10]:
sample_meta = sample_meta[keep_meta]

## Read labels and doublet calls from HISE

In [11]:
## search_id values from previous steps
label_search_id = 'zinc-bismuth-bismuth'
doublet_search_id = 'chlorine-sulfur-silver'

search_string = '|'.join([label_search_id, doublet_search_id])

Retrieve files stored in our HISE project store

In [12]:
ps_df = hisepy.list_files_in_project_store('UCSDCU_Y4')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [13]:
search_df = ps_df[ps_df['name'].str.contains(search_string)]
search_df = search_df[search_df['name'].str.contains('.parquet')]
search_df = search_df.sort_values('name')

In [14]:
search_df['name'].tolist()

['chlorine-sulfur-silver/ra_scrublet_results_2024-05-10.parquet',
 'zinc-bismuth-bismuth/ra_celltypist_AIFI_L1_2024-05-10.parquet',
 'zinc-bismuth-bismuth/ra_celltypist_AIFI_L2_2024-05-10.parquet',
 'zinc-bismuth-bismuth/ra_celltypist_AIFI_L3_2024-05-10.parquet']

In [15]:
search_df

,id,name
350,5928a13e-8086-49c0-9b8e-df43ca2cfb39,chlorine-sulfur-silver/ra_scrublet_results_202...
352,23abfd47-0d3f-4ca0-88fb-b2b0820f76b1,zinc-bismuth-bismuth/ra_celltypist_AIFI_L1_202...
354,2a6abb94-1628-4307-9abb-69127295ff8a,zinc-bismuth-bismuth/ra_celltypist_AIFI_L2_202...
356,c922d7ba-c596-4f8e-b1df-43bfaeeb0178,zinc-bismuth-bismuth/ra_celltypist_AIFI_L3_202...


## Read and combine label sets to simplify merges

In [17]:
label_list = []
for uuid in search_df['id']:
    res = read_parquet_uuid(uuid)
    res = res.set_index('barcodes', drop = True)
    label_list.append(res)

downloading fileID: 23abfd47-0d3f-4ca0-88fb-b2b0820f76b1
Files have been successfully downloaded!
downloading fileID: 2a6abb94-1628-4307-9abb-69127295ff8a
Files have been successfully downloaded!
downloading fileID: c922d7ba-c596-4f8e-b1df-43bfaeeb0178
Files have been successfully downloaded!


In [18]:
all_labels = pd.concat(label_list, axis = 1)

In [19]:
all_labels = all_labels.reset_index(drop = False)
all_labels.head()

,barcodes,predicted_doublet,doublet_score,AIFI_L1,over_clustering,majority_voting,AIFI_L1_score,AIFI_L2,over_clustering,majority_voting,AIFI_L2_score,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,063d9a88b99611ed84f6d6357bc258e4,False,0.060711,T cell,103,T cell,0.999960,Memory CD4 T cell,103,Memory CD4 T cell,0.856824,CM CD4 T cell,103,CM CD4 T cell,1.0
1,063d9fa6b99611ed84f6d6357bc258e4,False,0.180698,T cell,33,T cell,0.999996,Memory CD4 T cell,33,Memory CD4 T cell,0.946788,CM CD4 T cell,33,CM CD4 T cell,1.0
2,063da5b4b99611ed84f6d6357bc258e4,False,0.043691,Monocyte,39,Monocyte,0.999460,CD14 monocyte,39,CD14 monocyte,0.999232,Core CD14 monocyte,39,Core CD14 monocyte,1.0
3,063dac30b99611ed84f6d6357bc258e4,False,0.013381,T cell,7,T cell,0.999888,Memory CD4 T cell,7,Memory CD4 T cell,0.933583,KLRF1- GZMB+ CD27- memory CD4 T cell,7,KLRF1- GZMB+ CD27- memory CD4 T cell,1.0
4,063dad8eb99611ed84f6d6357bc258e4,False,0.059326,T cell,114,T cell,0.999965,Memory CD4 T cell,114,Memory CD4 T cell,0.954982,CM CD4 T cell,114,CM CD4 T cell,1.0


## Define subsets

In [22]:
subset_column = 'subset_grp'

grps = ["set1", "set2", "set3", "set4"]

num_repeats = len(sample_meta) // len(grps)

assigned_values = np.repeat(grps, num_repeats)

np.random.seed(202405)
np.random.shuffle(assigned_values)

sample_meta.loc[:, subset_column] = assigned_values

In [24]:
subset_counts = sample_meta[subset_column].value_counts()
subset_counts

subset_grp
set2    128
set4    128
set3    128
set1    128
Name: count, dtype: int64

In [26]:
subset_sample_meta = sample_meta.groupby(subset_column)

In [27]:
sample_meta.head()

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.race,subject.ethnicity,subject.birthYear,sample.sampleKitGuid,sample.visitName,sample.drawDate,sample.diseaseStatesRecordedAtVisit,sample.daysSinceFirstVisit,file.id,subset_grp
0,CU1,CU1057,Female,Caucasian,Non-Hispanic origin,1957,KT02989,Flu Year 1 Day 90,2022-01-01T00:00:00Z,Rheumatoid arthritis,120,41caa9d3-12e4-432b-8352-aa012905b823,set2
1,CU1,CU1051,Male,Caucasian,Non-Hispanic origin,1949,KT04080,RA Year 4 Visit 1,2022-08-01T00:00:00Z,Rheumatoid arthritis,460,d5720dff-ea39-4b45-9e45-b7459b779ee6,set4
2,CU1,CU1052,Female,Caucasian,Non-Hispanic origin,1988,KT03932,RA Year 4 Visit 1,2022-08-01T00:00:00Z,Rheumatoid arthritis,458,a2829968-97f2-48c8-bc0a-2989850aebf1,set3
3,CU1,CU1051,Male,Caucasian,Non-Hispanic origin,1949,KT04098,Flu Year 2 Stand-Alone,2022-03-01T00:00:00Z,Rheumatoid arthritis,315,d75a9be9-7092-4915-8532-b75c99a885fe,set4
4,CU1,CU1058,Female,Asian,Non-Hispanic origin,1969,KT04095,RA Year 4 Visit 1,2022-07-01T00:00:00Z,Rheumatoid arthritis,350,6b48b8b3-6848-4f2f-bc63-c2ccd7819948,set3


## Read and assemble anndata objects for each subset

In [28]:
group_input_dict = {}
group_adata_dict = {}
for group, df in subset_sample_meta:
    group_name = group
    #df = df.iloc[0:5] # Remove for full run
    group_uuids = df['file.id'].tolist()
    group_input_dict[group_name] = group_uuids

    # Cache files
    cache_dir = '/home/jupyter/cache'
    cache_paths = []
    for uuid in group_uuids:
        cache_path = '{d}/{u}'.format(d = cache_dir, u = uuid)
        if not os.path.isdir(cache_path):
            hise_res = hisepy.reader.cache_files([uuid])
        cache_paths.append(cache_path)

    # Get cached file paths
    cache_files = []
    for cache_path in cache_paths:
        fn = os.listdir(cache_path)[0]
        cache_files.append('{d}/{f}'.format(d = cache_path, f = fn))

    # Read cached files as anndata
    adata_list = []
    for cache_file in cache_files:
        adata = read_h5_anndata(cache_file)
        adata_list.append(adata)
    group_adata = sc.concat(adata_list)
    
    group_adata_dict[group_name] = group_adata

downloading fileID: 366ef40a-2241-4852-a985-14f9ef616964
Files have been successfully downloaded!
downloading fileID: 6b383ac8-f09b-446b-9366-c71090966909
Files have been successfully downloaded!
downloading fileID: ddbb6500-e453-4ad1-9d0b-874efbbea637
Files have been successfully downloaded!
downloading fileID: 58a68e43-cc40-4b5a-a4b5-5fe9fe73efd1
Files have been successfully downloaded!
downloading fileID: bf43dfa7-ad18-4edd-8eef-1d8740c208fb
Files have been successfully downloaded!
downloading fileID: 64aaa507-69cf-4de4-a0be-1c97e97deffb
Files have been successfully downloaded!
downloading fileID: 5215eda2-0cb0-4008-94f6-9f6ee6d31d71
Files have been successfully downloaded!
downloading fileID: 855aeafb-a170-4365-b055-4c0fdc902379
Files have been successfully downloaded!
downloading fileID: d627d5ad-d847-4e95-8ba6-a6ff2ebe8c16
Files have been successfully downloaded!
downloading fileID: 10c894bd-01e0-4be0-be41-810f45b13eb1
Files have been successfully downloaded!
downloading fileID: 

In [29]:
group_adata_dict.keys()

dict_keys(['set1', 'set2', 'set3', 'set4'])

In [30]:
for group_name, adata in group_adata_dict.items():
    print('{g}: {n} cells'.format(g = group_name, n = adata.shape[0]))

set1: 2087541 cells
set2: 2131497 cells
set3: 2143586 cells
set4: 2157122 cells


In [31]:
sum(adata.shape[0] for adata in group_adata_dict.values())

8519746

## Update Observations with additional metadata

Now, we'll add the sample metadata, CMV status, and BMI data to our scRNA-seq data.

First, we'll convert `pbmc_sample_id` to `sample.sampleKitGuid` using a regular expression. PBMC samples are derived from kits in our LIMS system, so both share the same numerical core. The difference is that there can be multiple PBMC samples collected at the same time, so PBMC samples are prefixed with PB to indicate their sample type, and suffixed with -XX to indicate an aliquot number.

In [32]:
def sample_to_kit(sample):
    kit = re.sub('PB([0-9]+)-.+','KT\\1',sample)
    return(kit)

To keep things tidy, we'll also drop the `seurat_pbmc_type`, `seurat_pbmc_type_score`, and UMAP coordinates generated by our processing pipeline. These cell type assignments are from a now-outdated reference dataset, and the UMAP coordinates are generated for viewing individual samples - not helpful for our full dataset.

In [33]:
drop_columns = [
    'seurat_pbmc_type','seurat_pbmc_type_score',
    'umap_1', 'umap_2'
]

Then, we'll add our new sample data with a left join on the `sample.sampleKitGuid` values, and add our labels and doublet calls with a left join on cell `barcodes`.

Next, we'll convert all of our text columns to categorical. This is used to make storage of text data more efficient when we write our output file, as we'll only need to store a single instance of our strings.

We do this for all columns except barcodes, which we need to retain as a string type for use as an index.

Finally, we'll add these back to our anndata object

In [34]:
for group_name, adata in group_adata_dict.items():
    print('{g}: {p}'.format(g = group_name, p = adata.obs['pbmc_sample_id'].str.startswith('PB').all()))

set1: True
set2: True
set3: True
set4: True


In [35]:
for group_name, adata in group_adata_dict.items():
    print(group_name)
    obs = adata.obs
    
    # Convert sample.sampleKitGuid
    print('converting ids')
    obs['sample.sampleKitGuid'] = [sample_to_kit(sample) for sample in obs['pbmc_sample_id']]
    
    # Drop old columns
    obs = obs.drop(drop_columns, axis = 1)
    
    print('merging sample metadata')
    # Add new metadata
    obs = obs.merge(
        sample_meta,
        how = 'left',
        on = 'sample.sampleKitGuid'
    )
    
    print('merging labels')
    # Add labels
    obs = obs.merge(all_labels, how = 'left', on = 'barcodes')
    
    print('converting to categorical')
    # Convert to categorical
    cat_obs = obs
    for i in range(cat_obs.shape[1]):
        col_name = cat_obs.dtypes.index.tolist()[i]
        col_type = cat_obs.dtypes[col_name]
        if col_name == 'barcodes':
            cat_obs[col_name] = cat_obs[col_name].astype(str)
        elif is_object_dtype(col_type):
            cat_obs[col_name] = cat_obs[col_name].astype('category')
    cat_obs = cat_obs.set_index('barcodes', drop = False)
    
    # Assign final observations back to anndata
    adata.obs = cat_obs
    
    group_adata_dict[group_name] = adata

set1
converting ids
merging sample metadata
merging labels
converting to categorical
set2
converting ids
merging sample metadata
merging labels
converting to categorical
set3
converting ids
merging sample metadata
merging labels
converting to categorical
set4
converting ids
merging sample metadata
merging labels
converting to categorical


## Write assembled data to disk

In [36]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [37]:
out_h5ads = {}
for group_name, adata in group_adata_dict.items():
    out_h5ad = 'output/ra_pbmc_{g}_raw_labeled_{d}.h5ad'.format(g = group_name, d = date.today())
    adata.write_h5ad(out_h5ad)
    out_h5ads[group_name] = out_h5ad

In [39]:
out_csvs = {}
out_parquets = {}
for group_name, adata in group_adata_dict.items():
    obs = adata.obs
    
    out_csv = 'output/ra_pbmc_{g}_raw_labeled_meta_{d}.csv'.format(g = group_name, d = date.today())
    obs.to_csv(out_csv)
    out_csvs[group_name] = out_csv

    out_parquet = 'output/ra_pbmc_{g}_raw_labeled_meta_{d}.parquet'.format(g = group_name, d = date.today())

    obs = obs.loc[:, ~obs.columns.duplicated()] ## remove duplicate columns 
    obs.to_parquet(out_parquet)
    out_parquets[group_name] = out_parquet

## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [40]:
ss = hisepy.get_study_spaces()
print(ss[2]['name'])
print(ss[2]['id'])
study_space_uuid = ss[2]['id']

title = 'Labeled Raw scRNA-seq Assembly {d}'.format(d = date.today())

preRA cross-sectional + earlyRA + Longitudinal analysis
223de760-9624-45bd-aefe-ca24c75b1800


In [41]:
search_id = element_id()
search_id

'radium-uranium-titanium'

In [42]:
in_files = [sample_meta_uuid] + search_df['id'].tolist()
in_files = in_files + sample_meta['file.id'].tolist()

In [43]:
len(in_files)

517

In [44]:
in_files[0:10]

['2b3673ee-827b-415e-837b-f27a0b88eae2',
 '5928a13e-8086-49c0-9b8e-df43ca2cfb39',
 '23abfd47-0d3f-4ca0-88fb-b2b0820f76b1',
 '2a6abb94-1628-4307-9abb-69127295ff8a',
 'c922d7ba-c596-4f8e-b1df-43bfaeeb0178',
 '41caa9d3-12e4-432b-8352-aa012905b823',
 'd5720dff-ea39-4b45-9e45-b7459b779ee6',
 'a2829968-97f2-48c8-bc0a-2989850aebf1',
 'd75a9be9-7092-4915-8532-b75c99a885fe',
 '6b48b8b3-6848-4f2f-bc63-c2ccd7819948']

In [45]:
out_files = list(out_h5ads.values()) + list(out_csvs.values()) + list(out_parquets.values())

In [46]:
out_files

['output/ra_pbmc_set1_raw_labeled_2024-05-10.h5ad',
 'output/ra_pbmc_set2_raw_labeled_2024-05-10.h5ad',
 'output/ra_pbmc_set3_raw_labeled_2024-05-10.h5ad',
 'output/ra_pbmc_set4_raw_labeled_2024-05-10.h5ad',
 'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.csv',
 'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.csv',
 'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.csv',
 'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.csv',
 'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.parquet',
 'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.parquet',
 'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.parquet',
 'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.parquet']

In [47]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/ra_pbmc_set1_raw_labeled_2024-05-10.h5ad', 'output/ra_pbmc_set2_raw_labeled_2024-05-10.h5ad', 'output/ra_pbmc_set3_raw_labeled_2024-05-10.h5ad', 'output/ra_pbmc_set4_raw_labeled_2024-05-10.h5ad', 'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.csv', 'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.csv', 'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.csv', 'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.csv', 'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.parquet', 'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.parquet', 'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.parquet', 'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.parquet']. Do you truly want to proceed?


(y/n) y


{'trace_id': '7d287404-118a-485f-a703-b991c299a517',
 'files': ['output/ra_pbmc_set1_raw_labeled_2024-05-10.h5ad',
  'output/ra_pbmc_set2_raw_labeled_2024-05-10.h5ad',
  'output/ra_pbmc_set3_raw_labeled_2024-05-10.h5ad',
  'output/ra_pbmc_set4_raw_labeled_2024-05-10.h5ad',
  'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.csv',
  'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.csv',
  'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.csv',
  'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.csv',
  'output/ra_pbmc_set1_raw_labeled_meta_2024-05-10.parquet',
  'output/ra_pbmc_set2_raw_labeled_meta_2024-05-10.parquet',
  'output/ra_pbmc_set3_raw_labeled_meta_2024-05-10.parquet',
  'output/ra_pbmc_set4_raw_labeled_meta_2024-05-10.parquet']}

In [48]:
import session_info
session_info.show()

#### Notes
1. removing majority voting and over clustering columns, rename these 2 columns based on levels?

In [50]:
all_labels.head()

,barcodes,predicted_doublet,doublet_score,AIFI_L1,over_clustering,majority_voting,AIFI_L1_score,AIFI_L2,over_clustering,majority_voting,AIFI_L2_score,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,063d9a88b99611ed84f6d6357bc258e4,False,0.060711,T cell,103,T cell,0.999960,Memory CD4 T cell,103,Memory CD4 T cell,0.856824,CM CD4 T cell,103,CM CD4 T cell,1.0
1,063d9fa6b99611ed84f6d6357bc258e4,False,0.180698,T cell,33,T cell,0.999996,Memory CD4 T cell,33,Memory CD4 T cell,0.946788,CM CD4 T cell,33,CM CD4 T cell,1.0
2,063da5b4b99611ed84f6d6357bc258e4,False,0.043691,Monocyte,39,Monocyte,0.999460,CD14 monocyte,39,CD14 monocyte,0.999232,Core CD14 monocyte,39,Core CD14 monocyte,1.0
3,063dac30b99611ed84f6d6357bc258e4,False,0.013381,T cell,7,T cell,0.999888,Memory CD4 T cell,7,Memory CD4 T cell,0.933583,KLRF1- GZMB+ CD27- memory CD4 T cell,7,KLRF1- GZMB+ CD27- memory CD4 T cell,1.0
4,063dad8eb99611ed84f6d6357bc258e4,False,0.059326,T cell,114,T cell,0.999965,Memory CD4 T cell,114,Memory CD4 T cell,0.954982,CM CD4 T cell,114,CM CD4 T cell,1.0


In [52]:
all_labels.shape

(8519746, 15)